# 07 RAG Retrieval Engine

This notebook implements the Retrieval-Augmented Generation (RAG) data pipeline.

**Updated approach:** SHAP explanations are computed for **all 4,508 patients** (not just the test split) 
so that the RAG pipeline can serve every patient for whom clinical text is available.
The train/test split is still used strictly for model *evaluation* metrics (AUC, F1, etc.), but after validation,
the model is retrained and run on all data to ensure maximum pipeline coverage (~1,192 matched patients instead of 241).

> **Note:** For training-set patients, SHAP values reflect in-sample model behaviour and may
> be slightly optimistic compared to held-out test patients. This is a common and acceptable standard for clinical 
> decision-support systems where every patient deserves a personalised explanation.

**Steps:**
1. Load all patients from `model_features`.
2. Temporarily retrain the model and save `model_artifacts.pkl` to guarantee sklearn API version compatibility (`multi_class`).
3. Compute SHAP values for **all** patients.
4. Load the clinical notes (`HPI.json` and `RAG_data_summary.json`).
5. Match all patients against RAG data and build enriched prompt contexts.
6. Save all generated prompts to `rag_prompts.json`.

In [ ]:
import duckdb
import pandas as pd
import numpy as np
import pickle
import json
import shap
import warnings
import os
warnings.filterwarnings('ignore')

from utils import get_db_connection

In [ ]:
# ── Step 1: Load ALL patients from DB and extract features ───────────────────
print("Connecting to DB to load model_features...")
con = get_db_connection()
df = con.execute("SELECT * FROM model_features").fetchdf()
con.close()
print(f"Loaded {len(df)} total patients from model_features.")

# Add lab CHANGE features (last - first)
LABS = [
    'creatinine', 'urea_nitrogen', 'sodium', 'potassium', 'glucose',
    'hemoglobin', 'white_blood_cells', 'platelet_count', 'bicarbonate',
    'calcium_total', 'inrpt', 'ptt', 'troponin_t',
    'creatine_kinase_mb_isoenzyme'
]
for lab in LABS:
    df[f'{lab}_change'] = df[f'{lab}_last'] - df[f'{lab}_first']

# Fill missing numerical values with median
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
for c in ['readmitted_30d', 'hadm_id', 'subject_id']:
    if c in num_cols:
        num_cols.remove(c)
for col in num_cols:
    df[col] = df[col].fillna(df[col].median())

# Fill and encode categorical columns
cat_cols = ['gender', 'admission_type', 'insurance', 'marital_status', 'race', 'discharge_location']
for col in cat_cols:
    df[col] = df[col].fillna('UNKNOWN')

df_encoded = pd.get_dummies(df, columns=cat_cols, drop_first=True)

# Separate features, labels, and id indices
all_hadm_ids = df_encoded['hadm_id'].values
X_all = df_encoded.drop(columns=['subject_id', 'hadm_id', 'readmitted_30d'])
y_all = df_encoded['readmitted_30d']

print(f"Feature matrix shape (all patients): {X_all.shape}")

In [ ]:
# ── Step 2: Retrain model slightly to ensure correct scikit-learn version ────
# (Fixes `multi_class` error by refreshing the saved model_artifacts.pkl)

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_all, test_size=0.2, random_state=42, stratify=y_all)

selector = SelectKBest(f_classif, k=40)
X_train_sel = selector.fit_transform(X_train, y_train)
X_test_sel  = selector.transform(X_test)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train_sel)
X_test_sc  = scaler.transform(X_test_sel)

selected_features = X_all.columns[selector.get_support()].tolist()

model = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
model.fit(X_train_sc, y_train)

print(f"Model retrained. Evaluation Test AUC: {roc_auc_score(y_test, model.predict_proba(X_test_sc)[:,1]):.4f}")

# Save artifacts securely
artifacts = {
    'scaler': scaler,
    'selector': selector,
    'model': model,
    'selected_features': selected_features
}
artifacts_path = '../model_artifacts.pkl'
with open(artifacts_path, 'wb') as f:
    pickle.dump(artifacts, f)
print("Fresh model_artifacts.pkl saved.")

In [ ]:
# ── Step 3: Compute SHAP for ALL patients ────────────────────────────────────
X_all_sel = selector.transform(X_all)
X_all_sc  = scaler.transform(X_all_sel)
X_all_df  = pd.DataFrame(X_all_sc, columns=selected_features)

print(f"Computing SHAP values for all {len(X_all_df)} patients...")
explainer   = shap.LinearExplainer(model, X_all_df)
shap_values = explainer(X_all_df)
print("SHAP values calculated successfully.")

# Get predicted probabilities for all patients
y_pred_proba_all = model.predict_proba(X_all_df)[:, 1]
print(f"Risk scores computed for {len(y_pred_proba_all)} patients.")

In [ ]:
# ── Step 4: Load RAG Data ────────────────────────────────────────────────────
rag_data_path = '../dataset/RAG_data_summary.json'
with open(rag_data_path, 'r') as f:
    rag_list = json.load(f)

# Convert to dictionaries for O(1) lookup based on hadm_id strictly
rag_data = {list(item.keys())[0]: list(item.values())[0] for item in rag_list}
print(f"Loaded {len(rag_data)} RAG summaries from original JSON.")

In [ ]:
# ── Step 5: RAG Retrieval Function (uses all-patient index) ──────────────────
# Build a hadm_id → index map for fast lookup in probabilities array
hadm_id_to_idx = {str(hid): i for i, hid in enumerate(all_hadm_ids)}

def get_patient_context(hadm_id_str, num_features=3):
    if hadm_id_str not in hadm_id_to_idx:
        return f"Error: Patient {hadm_id_str} not found in the dataset."

    idx = hadm_id_to_idx[hadm_id_str]

    # Get Risk Score
    risk_prob = y_pred_proba_all[idx]

    # Get Top SHAP Model Drivers
    patient_shap  = shap_values.values[idx]
    feature_names = shap_values.feature_names
    sorted_idx    = np.argsort(patient_shap)[::-1]
    top_drivers   = [(feature_names[i], patient_shap[i]) for i in sorted_idx[:num_features] if patient_shap[i] > 0]

    drivers_str = "\n".join([f"- {feat} (Impact score: {val:.4f})" for feat, val in top_drivers])
    if not drivers_str:
        drivers_str = "No significant risk-increasing factors found."

    # Get Clinical summary
    clinical_summary = rag_data.get(hadm_id_str, "No clinical summary available.")

    prompt_context = f"""--- PATIENT CLINICAL HISTORY ---
{clinical_summary}

--- AI MODEL PREDICTION ---
Predicted Readmission Risk: {risk_prob:.1%}

--- KEY RISK DRIVERS (SHAP) ---
The model identified the following key factors driving this risk:
{drivers_str}
"""
    return prompt_context


def generate_explanation_placeholder(prompt_context):
    """Placeholder for the LLM API generation coming up in Notebook 08."""
    return "LLM Generation Pending... (Execute Notebook 08 to generate predictions)"


# Pre-match IDs before generation
rag_ids = set(rag_data.keys())
all_matched_ids = [hid for hid in all_hadm_ids if str(hid) in rag_ids]
print(f"Users that matched across DB & JSON: {len(all_matched_ids)} out of {len(all_hadm_ids)}")

# Demo: show full prompt context for 3 matched patients
for hadm_id in all_matched_ids[:3]:
    print("\n" + "="*70)
    print(f"PATIENT hadm_id: {hadm_id}")
    print("="*70)
    context = get_patient_context(str(hadm_id))
    print(context)
    print("--- [PLACEHOLDER LLM OUTPUT] ---")
    print(generate_explanation_placeholder(context))

In [ ]:
# ── Step 6: Package output to rag_prompts.json ──────────────────────────────
output_path = '../rag_prompts.json'

rag_output = []
for hadm_id in all_matched_ids:
    ctx = get_patient_context(str(hadm_id))
    rag_output.append({'hadm_id': int(hadm_id), 'prompt_context': ctx})

with open(output_path, 'w') as f:
    json.dump(rag_output, f, indent=2)

print(f"\nFINAL STATUS:\nSaved {len(rag_output)} generated prompt contexts for the LLM to process to rag_prompts.json.")
print(f"\nCOVERAGE METRICS:")
print(f"  Total patients in database:         {len(all_hadm_ids)}")
print(f"  Patients matched w/ Clinical text:  {len(all_matched_ids)}")
print(f"  Final AI pipeline matching density: {100*len(all_matched_ids)/len(all_hadm_ids):.1f}%")